# Evaluation protocol

## Importing modules

In [ ]:
import numpy as np
import pandas as pd
from sksurv.util import Surv
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw

## Importing data

In [ ]:
clinical_dataset = pd.read_csv('../data/processed/clinical_train_processed.csv', index_col=0)
clinical_dataset.head()

Then we are splitting the labels and the features. The labels are stored in Y and the features in X_clinical

In [ ]:
Y = clinical_dataset[['OS_YEARS', 'OS_STATUS']]
print(f"Shape of the labels dataset: {Y.shape}")
Y.head()

In [ ]:
X_clinical = clinical_dataset.drop(['OS_YEARS', 'OS_STATUS'], axis=1)
print(f"Shape of the features dataset: {X_clinical.shape}")
X_clinical.head()

## Transforming Y into a structured array

Survival model takes structured array with the status column (boolean) and the time column.

In [ ]:
y = Surv.from_arrays(event= Y['OS_STATUS'], time= Y['OS_YEARS'])
y

## Splitting the datasets into training and validation set

We are splitting the datasets using StratifiedKfold to ensure the same proportion of censored and uncensored patients in the training and validation sets.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

status = Y['OS_STATUS'].astype(int)
for train_idx, val_idx in skf.split(X_clinical, Y['OS_STATUS']):
    print(f"Train: {np.bincount(status.iloc[train_idx])}, Val: {np.bincount(status.iloc[val_idx])}")

Verifying if all patients are in one and one time only in a validation.

In [ ]:
folds = list(skf.split(X_clinical, status))
all_val_idx = np.concatenate([val_idx for _, val_idx in folds])

print(len(all_val_idx))                    # 3173 (all patients, one time each)
print(len(np.unique(all_val_idx)))         # 3173 (unique)
print(np.array_equal(np.sort(all_val_idx), np.arange(len(X_clinical))))  # True (exact coverage)

## IPCW C-index metrics

We are making a function that return the C-index between the prediction and the validation.
- risk_scores is the prediction of our model
- y_train is used to established the weight of the index (to be able to use the censored patients)

In [ ]:
def ipcw_cindex(y_train, y_val, risk_scores, tau=7.0):
    """Calculate the C-index between the prediction and the validation fold"""
    C_index = concordance_index_ipcw(y_train, y_val, risk_scores, tau=tau)[0]
    return C_index

Test of the function. A perfect prediction returns 1.0, a perfectly inverse prediction returns 0.0 and a random prediction returns 0.5.

In [ ]:
toy = pd.DataFrame({"event": [True,  True,  False, False], "time":  [1.0,   4.0,   2.0,   6.0],})
y_toy = Surv.from_arrays(event=toy["event"], time=toy["time"])
risk_scores_toy_1 = np.array([100, 70, 20, 50]) # perfect prediction
risk_scores_toy_2 = np.array([20, 50, 100, 70]) # inverse prediction
risk_scores_toy_3 = np.array([42, 42, 42, 42]) # random prediction
risk_scores_toy_4 = np.array([50, 100, 70, 20])
print(f"Prediction 1: {ipcw_cindex(y_toy, y_toy, risk_scores_toy_1)}, prdiction 2: {ipcw_cindex(y_toy, y_toy, risk_scores_toy_2)}, prediction 3: {ipcw_cindex(y_toy, y_toy, risk_scores_toy_3)}, prediction 4: {ipcw_cindex(y_toy, y_toy, risk_scores_toy_4)}.")

## Baseline : BM_BLAST

We are making an evaluation function to test a risk scores calculated on the BM_BLAST measure.The level of blast is indeed linked to the severity of the deceases.

In [ ]:
fold_score = []
for train_idx, val_idx in folds:
    # replacing the nan by the median
    med = X_clinical['BM_BLAST_LOG'].iloc[train_idx].median()

    # estimated the risk score by the level of blasts
    baseline_scores = X_clinical['BM_BLAST_LOG'].iloc[val_idx].fillna(med).to_numpy()

    # calculating the C-index for the fold
    c=ipcw_cindex(y[train_idx], y[val_idx], baseline_scores)
    fold_score.append(c)

print(f'Baseline BM_BLAST C-index: {np.mean(fold_score)} ± {np.std(fold_score)}')

The baseline on the BM_BLAST shows a C-index of 0.616 ± 0.009.